In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

# NL2SQL — LangGraph Agentic Pipeline

This notebook refactors the original agentic retry loop into a **LangGraph StateGraph** — replacing an implicit Python for-loop with explicit nodes, edges, and state management.

**Result: 85.59% execution accuracy** on Spider validation set — identical to the original agent, now with cleaner architecture and explicit state flow.

---

## Pipeline Architecture

```
Initial State (index, db_id, question)
         │
         ▼
  ┌─────────────┐
  │ build_prompt │  ← builds agentic prompt with schema + CoT scaffold
  │             │    on retry: appends failed SQL + error message
  └──────┬──────┘
         │
         ▼
  ┌─────────────┐
  │ generate_sql │  ← Llama 3.1 8B generates SQL via chain-of-thought
  │             │    extract_sql_from_reasoning() takes the last SQL block
  └──────┬──────┘
         │
         ▼
  ┌─────────────┐
  │ execute_sql  │  ← runs SQL against SQLite database
  │             │    returns result, error string, or None
  └──────┬──────┘
         │
         ▼
  ┌─────────────────────────────┐
  │        should_retry?         │
  │  error AND retry_count < 4  │
  └──────┬──────────────────────┘
         │                    │
      YES │                NO  │
         │                    ▼
         │                   END
         │                (return final_sql)
         │
         └──→ back to build_prompt (retry_count += 1)
```

State flows explicitly through every node via `AgentState` — no hidden variables, no implicit control flow. Each node reads what it needs and returns only what changed.

---

## What This Notebook Does

1. Loads the fine-tuned Llama 3.1 8B QLoRA adapter (trained in the [Training Notebook](https://www.kaggle.com/code/mehulkumar99/spider-question-to-sql-query))
2. Runs the LangGraph agent on the Spider validation set wrong + error examples (197 total)
3. Self-corrects SQL via a conditional retry loop — up to 4 attempts per example
4. Evaluates execution accuracy against gold SQL on the full 1034 validation examples

---

## nl2sql-modules Dataset

Utility and evaluation functions are modularized into two scripts, available as a public Kaggle dataset ([nl2sql-modules](https://www.kaggle.com/datasets/mehulkumar99/nl2sql-modules)):

- **`nl2sql_utils.py`** — agent functions: `build_agentic_prompt`, `generate`, `extract_sql_from_reasoning`, `execute_sql`
- **`nl2sql_evals.py`** — evaluation functions: `normalize_df`, `execution_match`, `evaluate_pair`, `execution_accuracy_parallel`

---

## Related Notebooks

| Notebook | Description |
|---|---|
| [Training Notebook](https://www.kaggle.com/code/mehulkumar99/spider-question-to-sql-query) | QLoRA fine-tuning on augmented Spider schema |
| [Inference Notebook](https://www.kaggle.com/code/mehulkumar99/inference-notebook-for-question-to-sql-query) | Batched inference + execution accuracy evaluation |
| [Original Agent Notebook](https://www.kaggle.com/code/mehulkumar99/agentic-loop-addressing-syntax-errors) | Python-based agentic retry loop (84.14% → 85.59%) |
| [RAG Notebook](https://www.kaggle.com/code/mehulkumar99/rag-sql) | Column-level RAG for schema pruning |
| [Blog](https://www.kaggle.com/code/mehulkumar99/blog-question-to-sql-query) | Full project writeup with failure analysis |

In [2]:
import os
print(os.path.exists('/kaggle/working/nl2sql_utils.py'))

True


In [3]:
!pip install langgraph langchain

# Pip Install

In [4]:
import os
os.environ['LD_LIBRARY_PATH'] = '/usr/local/lib/python3.12/dist-packages/nvidia/cu13/lib:' + os.environ.get('LD_LIBRARY_PATH', '')

!pip install --upgrade huggingface_hub -q
!pip install "torchvision>=0.27.0" --upgrade -q
!pip install "unsloth @ git+https://github.com/unslothai/unsloth.git" -q
!pip install "unsloth_zoo @ git+https://github.com/unslothai/unsloth-zoo.git" --force-reinstall --no-cache-dir -q
!pip install --no-deps "trl<0.13.0" peft accelerate -q

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.7/58.7 kB 35.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 222.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 97.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 210.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.2/94.2 kB 179.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 29.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 217.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 693.4/693.4 kB 296.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 265.1 MB/s eta 0:00:00
  

In [5]:
!pip install bitsandbytes --upgrade -q
import ctypes
ctypes.CDLL('/usr/local/lib/python3.12/dist-packages/nvidia/cu13/lib/libnvJitLink.so.13')
import bitsandbytes

In [30]:
import sys
sys.path.append('/kaggle/input/nl2sql-modules/')

from nl2sql_utils import build_agentic_prompt, generate, execute_sql, extract_sql_from_reasoning
from nl2sql_evals import     normalize_df, execution_match, evaluate_pair, evaluate_single, execution_accuracy_parallel

# Loading Model

In [13]:
import os
from huggingface_hub import snapshot_download

# This is a safe, visible directory
local_model_path = "/kaggle/working/llama-3-1-clean"

snapshot_download(
    repo_id="unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    local_dir=local_model_path,
    local_dir_use_symlinks=False, # FORCES real files, no broken pointers
    resume_download=True
)

from unsloth import FastLanguageModel
import torch
from peft import PeftModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name   = "/kaggle/working/llama-3-1-clean", # No more cache!
    max_seq_length = 2048,
    dtype        = None,
    load_in_4bit = True,
    device_map = {"":0} # multi-GPU sharding is making your inference slower. forcing everything on cuda:0
) 




/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:189: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


W0612 06:07:19.898000 485 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0612 06:07:19.934000 485 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


🦥 Unsloth Zoo will now patch everything to make training faster!


<string>:1: FutureWarning: torch._dynamo.config.inline_inbuilt_nn_modules is deprecated and does not do anything, inline_inbuilt_nn_modules is always True. It will be removed in a future version of PyTorch.


==((====))==  Unsloth 2026.6.3: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.12.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.7.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load /kaggle/working/llama-3-1-clean as a legacy tokenizer.


In [14]:
# Apply LoRA adapters

adapter_path = '/kaggle/input/datasets/mehulkumar99/450-checkpoint-81/outputs/checkpoint-450'
model = PeftModel.from_pretrained(model, adapter_path)

print(f"Memory used: {model.get_memory_footprint()/1e9:.2f} GB")
model.print_trainable_parameters()

Memory used: 5.76 GB
trainable params: 0 || all params: 8,072,204,288 || trainable%: 0.0000


# Loading dataset

In [15]:
validation_df = pd.read_csv('/kaggle/input/datasets/mehulkumar99/81validation-df/81newest_validation_predictions (1).csv')
validation_df.head(3)

,db_id,query,question,schema,count_tables,predicted_sql,results,T,new_pred_sql
0,concert_singer,SELECT count(*) FROM singer,How many singers do we have?,"stadium : Stadium_ID (number) , Location (text...",4,SELECT count(*) FROM singer,correct,551,SELECT count(*) FROM singer
1,concert_singer,SELECT count(*) FROM singer,What is the total number of singers?,"stadium : Stadium_ID (number) , Location (text...",4,SELECT count(*) FROM singer,correct,552,SELECT count(*) FROM singer
2,concert_singer,"SELECT name , country , age FROM singer ORDE...","Show name, country, age for all singers ordere...","stadium : Stadium_ID (number) , Location (text...",4,"SELECT name, country, age FROM singer ORDER ...",correct,563,"SELECT name, country, age FROM singer ORDER ..."


In [16]:
# Isolate the problematic rows
error_rows =  validation_df[validation_df['results'] == 'error'].copy()
wrong_rows =  validation_df[validation_df['results'] == 'wrong'].copy()
# err_wro_rows = pd.concat([error_rows, wrong_rows], ignore_index = True)
err_wro_rows = validation_df[validation_df['results'].isin(['wrong', 'error'])].copy()


print(f"Total Errors: {len(error_rows)}")
print(f"Total Wrong: {len(wrong_rows)}")
err_wro_rows.head(3)

Total Errors: 36
Total Wrong: 161


,db_id,query,question,schema,count_tables,predicted_sql,results,T,new_pred_sql
16,concert_singer,"select max(capacity), average from stadium",What is the maximum capacity and the average o...,"stadium : Stadium_ID (number) , Location (text...",4,"SELECT max(capacity), avg(Average) FROM stadium",wrong,556,"SELECT max(capacity), avg(Average) FROM stadium"
43,concert_singer,select count(*) from concert where stadium_id ...,Find the number of concerts happened in the st...,"stadium : Stadium_ID (number) , Location (text...",4,SELECT count(*) FROM concert AS T1 JOIN stadiu...,wrong,558,SELECT count(*) FROM concert AS t1 JOIN stadiu...
44,concert_singer,select count(*) from concert where stadium_id ...,What are the number of concerts that occurred ...,"stadium : Stadium_ID (number) , Location (text...",4,SELECT count(*) FROM concert AS T1 JOIN stadiu...,wrong,560,SELECT count(*) FROM concert AS t1 JOIN stadiu...


In [17]:
error_indices = err_wro_rows.index.tolist()

# Loading schema

In [18]:
import pickle

# Open the file in read-binary mode
with open('/kaggle/input/datasets/mehulkumar99/augmented-schema/schema_lookup_augmented.pkl', 'rb') as file:
    schema_lookup = pickle.load(file)

# View the data
print(schema_lookup['concert_singer']['augmented_schema'])

# Table: stadium 
## Columns:
- Stadium_ID (number, PK) | e.g: [1, 2, 3]
- Location (text) | e.g: ['Raith Rovers', 'Ayr United', 'East Fife']
- Name (text) | e.g: ["Stark's Park", 'Somerset Park', 'Bayview Stadium']
- Capacity (number) | e.g: [10104, 11998, 2000]
- Highest (number) | e.g: [4812, 2363, 1980]
- Lowest (number) | e.g: [1294, 1057, 533]
- Average (number) | e.g: [2106, 1477, 864]

# Table: singer 
## Columns:
- Singer_ID (number, PK) | e.g: [1, 2, 3]
- Name (text) | e.g: ['Joe Sharp', 'Timbaland', 'Justin Brown']
- Country (text) | e.g: ['Netherlands', 'United States', 'France']
- Song_Name (text) | e.g: ['You', 'Dangerous', 'Hey Oh']
- Song_release_year (text) | e.g: ['1992', '2008', '2013']
- Age (number) | e.g: [52, 32, 29]
- Is_male (others) | e.g: ['F', 'T', 'T']

# Table: concert 
## Columns:
- concert_ID (number, PK) | e.g: [1, 2, 3]
- concert_Name (text) | e.g: ['Auditions', 'Super bootcamp', 'Home Visits']
- Theme (text) | e.g: ['Free choice', 'Free choice 2', 'Bl

# agent

In [19]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, Optional, List

class AgentState(TypedDict):
    index: int
    prompt: str
    generated_sql: str
    execution_result: Optional[list]
    execution_error: Optional[str]
    retry_count: int
    final_sql: str


In [20]:
def build_prompt_node(state: AgentState) -> dict:
    # read from state
    index = state["index"]
    error_msg = state["execution_error"]
    last_exec_sql = state["generated_sql"] if state["retry_count"] > 0 else None
    
    # calling existing function
    prompt_data = build_agentic_prompt(
        schema_lookup=schema_lookup,
        index=index,
        dataset=validation_df,
        last_exec_sql=last_exec_sql,
        error_msg=error_msg,
        inference=True
    )
    
    # return only what changed
    return {"prompt": prompt_data["text"]}

# output = build_prompt_node(initial_state)
# print(output['prompt'])
# print(prompt)

In [21]:
def generate_node(state: AgentState) -> dict:

    prompt = state['prompt']

    # calling existing functions
    output = generate(prompt, tokenizer, model)
    sql = extract_sql_from_reasoning(output)

    return {"generated_sql": sql}
    
    

In [22]:
def execute_sql_node(state: AgentState) -> dict:

    index = state['index']
    db_id = validation_df['db_id'].iloc[index]
    sql = state['generated_sql']

    # calling existing function
    KAGGLE_DB_DIR = '/kaggle/input/datasets/jeromeblanchet/yale-universitys-spider-10-nlp-dataset/spider/database'
    result = execute_sql(sql, db_id, db_dir = KAGGLE_DB_DIR )

    if isinstance(result, list):
        return {

            'execution_result': result,
            'final_sql' : sql,
            'execution_error': None, 
            'retry_count': state['retry_count']
}
    
    elif isinstance(result,str):
        return { 
            'execution_result': None,
            'execution_error': result,
            'final_sql': sql, # tracks last attempted sql regardless of success.
            'retry_count': state['retry_count'] +1
        }
    elif result is None:

        return{
            'execution_result': None,
            'execution_error': None,
            'final_sql': sql,
            'retry_count': state['retry_count']
        }

In [23]:
def should_retry(state: AgentState) -> str:

    if state["execution_error"] and state["retry_count"] < 4:
        return "build_prompt"
    return END # stop, return final state


        

In [24]:

graph = StateGraph(AgentState)

# register nodes: name → function
graph.add_node("build_prompt", build_prompt_node)
graph.add_node("generate_sql", generate_node)
graph.add_node("execute_sql", execute_sql_node)

# entry point
graph.set_entry_point("build_prompt")

# fixed edges
graph.add_edge("build_prompt", "generate_sql")
graph.add_edge("generate_sql", "execute_sql")

# conditional edge after execute_sql
graph.add_conditional_edges("execute_sql", should_retry)

# compile
app = graph.compile()

## test on one example

In [25]:
idx = error_indices[0]  # first error example

initial_state = {
    "index": idx,
    "prompt": "",
    "generated_sql": "",
    "execution_result": None,
    "execution_error": None,
    "retry_count": 0,
    "final_sql": ""
}

final_state = app.invoke(initial_state)

print("Final SQL:", final_state["final_sql"])
print("Retry count:", final_state["retry_count"])
print("Execution error:", final_state["execution_error"])
print("Execution result:", final_state["execution_result"])

Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.1

Final SQL: SELECT max(Capacity), avg(Average)
FROM STADIUM;
Retry count: 0
Execution error: None
Execution result: [(52500, 917.8888888888889)]


## running on all error and wrong indices

In [26]:
from tqdm.auto import tqdm

# Assuming error_indices is your list of indexes to fix
# Using tqdm(error_indices) will show the progress bar in your notebook
for idx in tqdm(error_indices, desc="Correcting SQL Hallucinations"):

    initial_state = {
    "index": idx,
    "prompt": "",
    "generated_sql": "",
    "execution_result": None,
    "execution_error": None,
    "retry_count": 0,
    "final_sql": ""
}

    final_state = app.invoke(initial_state)
    final_sql = final_state['final_sql']

    #Update the 'new_pred_sql' column with the corrected SQL
    validation_df.at[idx, 'new_pred_sql'] = final_sql
    

Correcting SQL Hallucinations:   0%|          | 0/197 [00:00<?, ?it/s]

Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_

SQL Error on pets_1: no such table: HAS_PETS


Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


SQL Error on pets_1: no such table: students


...

SQL Error on dog_kennels: no such column: tt.description


Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


SQL Error on dog_kennels: no such column: tt.description


Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


SQL Error on real_estate_properties: no such column: T2.property_type_description


In [27]:
KAGGLE_DB_DIR = '/kaggle/input/datasets/jeromeblanchet/yale-universitys-spider-10-nlp-dataset/spider/database'
accuracy, results = execution_accuracy_parallel(validation_df, max_workers=4, db_dir = KAGGLE_DB_DIR )

Evaluating:   0%|          | 0/1034 [00:00<?, ?it/s]

Execution Accuracy : 0.8559 (885/1034)
Wrong              : 126/1034
Invalid SQL errors : 23/1034


In [28]:
validation_df['results'] = results
validation_df.to_csv("/kaggle/working/CoT_agent_validation_predictions.csv", index=False)
validation_df['results'].value_counts()

results
correct    885
wrong      126
error       23
Name: count, dtype: int64